> PostgreSQL 14+ practice. Run `sql/00_sample_data.sql` first, then `SET search_path TO sql_practice, public;` in this connection. Run cells in order in the same SQL session. See README.md for setup and expected results. These are SQL cells; the original Python notebook metadata did not make them executable as Python.


## Sample database schema
We'll use a fintech database throughout.
Table. 	Columns
payment_transactions	transaction_id, customer_id, provider_id, amount, status, transaction_date
customers	customer_id, customer_name, country, created_at
payment_providers	provider_id, provider_name
refunds	refund_id, transaction_id, refund_amount, refund_date

### Conditional aggregation with success rate

In [ ]:
-- Count every status per provider; divide successful count by total using decimal arithmetic.
SELECT provider_id, COUNT(*) AS total_transactions,
    SUM(CASE WHEN status = 'Success' THEN 1 ELSE 0 END) AS successful_transactions,
    ROUND(100.0 * SUM(CASE WHEN status = 'Success' THEN 1 ELSE 0 END) / COUNT(*), 2) AS success_rate
FROM payment_transactions
GROUP BY provider_id
ORDER BY success_rate DESC, provider_id;

## Aggregation with multiple business conditions

In [ ]:
-- HAVING filters provider groups after aggregation; both volume thresholds must pass.
SELECT provider_id, COUNT(*) AS total_transactions,
    SUM(amount) AS total_spent, AVG(amount) AS average_transaction
FROM payment_transactions
GROUP BY provider_id
HAVING COUNT(*) > 5 AND SUM(amount) > 100000
ORDER BY total_spent DESC;

## Joining multiple tables

In [ ]:
-- LEFT JOIN enriches payments while keeping unmatched customer IDs; only successful payments remain.
SELECT t.provider_id, t.transaction_id, t.transaction_date,
    c.customer_name, c.country, p.provider_name, t.amount, t.status
FROM payment_transactions AS t
LEFT JOIN customers AS c ON t.customer_id = c.customer_id
LEFT JOIN payment_providers AS p ON t.provider_id = p.provider_id
WHERE t.status = 'Success'
ORDER BY t.transaction_date DESC, t.transaction_id DESC;